# Lecture 10: Fixed Exchange Rates

**Macroeconomics B -- Chapter 24**

This notebook lets you check the main mechanisms numerically:

1. how Denmark's nominal peg looks in the data,
2. how expected devaluation enters UIP,
3. how inflation differentials move the real exchange rate under a peg,
4. how the fixed-rate AS--AD model converges back to long-run equilibrium,
5. why temporary fiscal stimulus creates a later competitiveness hangover.

Run the notebook from top to bottom first. Then use the short exercises to change one assumption at a time.


**Table of contents**<a id='toc0_'></a>
- 1. [Setup](#toc1_)
- 2. [Denmark's nominal peg](#toc2_)
- 3. [UIP and credibility](#toc3_)
- 4. [Inflation differentials and real adjustment](#toc4_)
- 5. [Fixed-rate AS--AD dynamics](#toc5_)
- 6. [Fiscal policy under a peg](#toc6_)
- 7. [Anticipated devaluation](#toc7_)
- 8. [Group exercise](#toc8_)
- 9. [Summary](#toc9_)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
})

DATA = Path('data')
FIGS = Path('figs')


## 1. <a id='toc1_'></a>[Setup](#toc0_)

The parameter names follow the lecture notation. The model is written in deviations from long-run equilibrium:

$$
\widehat y_t=y_t-\bar y, \qquad \widehat\pi_t=\pi_t-\pi^f.
$$

The fixed-rate model is

$$
\widehat y_t=\beta_1(e^r_{t-1}-\widehat\pi_t)+z_t, \qquad
\widehat\pi_t=\gamma\widehat y_t+s_t, \qquad
 e^r_t=e^r_{t-1}-\widehat\pi_t.
$$

The exercise is useful because it turns the fixed-exchange-rate story into a simple dynamic system. Instead of only saying that inflation and competitiveness adjust over time, you can see how a shock today changes output, inflation, and the real exchange rate period by period.


In [ ]:
par = {
    'T': 30,
    'beta1': 0.70,   # demand sensitivity to the real exchange rate
    'gamma': 0.50,   # AS slope
    'beta3': 1.00,   # effect of government spending in the demand shock z
    'i_f': 0.02,     # foreign nominal interest rate, used in UIP exercises
}


def read_csv_series(filename):
    # Read a FRED-style CSV from the data folder.
    path = DATA / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Keep the notebook next to the data/ folder.")
    df = pd.read_csv(path)
    date_col = 'observation_date' if 'observation_date' in df.columns else 'date'
    value_col = [c for c in df.columns if c != date_col][0]
    s = pd.Series(pd.to_numeric(df[value_col], errors='coerce').values,
                  index=pd.to_datetime(df[date_col]),
                  name=filename.replace('.csv', ''))
    return s.dropna()


def yoy(series):
    return 100.0 * (series / series.shift(12) - 1.0)


def uip_rate(i_f, expected_depreciation):
    # Log-linear UIP: i = i_f + expected depreciation.
    return i_f + expected_depreciation


def simulate_fixed_rate(T, beta1, gamma, z_path=None, s_path=None,
                        fiscal_phi=0.0, beta3=1.0, er0=0.0):
    # Simulate the fixed-rate AS-AD model.
    # Fiscal rule, when fiscal_phi > 0: g_t - gbar = - fiscal_phi * yhat_t.
    if z_path is None:
        z_path = np.zeros(T)
    if s_path is None:
        s_path = np.zeros(T)
    z_path = np.asarray(z_path, dtype=float)
    s_path = np.asarray(s_path, dtype=float)
    yhat = np.empty(T)
    pihat = np.empty(T)
    er = np.empty(T)
    g_gap = np.empty(T)
    er_lag = er0
    for t in range(T):
        denom = 1.0 + beta3 * fiscal_phi + beta1 * gamma
        yhat[t] = (beta1 * er_lag + z_path[t] - beta1 * s_path[t]) / denom
        pihat[t] = gamma * yhat[t] + s_path[t]
        g_gap[t] = -fiscal_phi * yhat[t]
        er[t] = er_lag - pihat[t]
        er_lag = er[t]
    return pd.DataFrame({
        'period': np.arange(T),
        'z': z_path,
        'yhat': yhat,
        'pihat': pihat,
        'er': er,
        'g_gap': g_gap,
    })


## 2. <a id='toc2_'></a>[Denmark's nominal peg](#toc0_)

The Danish krone is pegged to the euro. The central rate is 7.46038 DKK per EUR and the formal narrow ERM II band is $\pm 2.25\%$. The next cell reconstructs DKK per EUR from USD cross rates available in the data folder.


In [ ]:
# DEXUSEU: USD per EUR, daily. EXDNUS: DKK per USD, monthly.
usd_per_eur = read_csv_series('DEXUSEU.csv').resample('MS').mean()
dkk_per_usd = read_csv_series('EXDNUS.csv').resample('MS').mean()
dkk_per_eur = (usd_per_eur * dkk_per_usd).dropna()
dkk_per_eur = dkk_per_eur.loc[dkk_per_eur.index >= '1999-01-01']

central = 7.46038
upper = central * 1.0225
lower = central * 0.9775

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(dkk_per_eur.index, dkk_per_eur, label='DKK per EUR')
ax.axhline(central, linestyle='--', label='central rate')
ax.axhline(upper, linestyle=':', label='+/- 2.25% band')
ax.axhline(lower, linestyle=':')
ax.set_title('DKK per EUR under ERM II')
ax.set_xlabel('Date')
ax.set_ylabel('DKK per EUR')
ax.legend(loc='upper right')
plt.show()

print(f"Last observation: {dkk_per_eur.index[-1].date()}, DKK/EUR = {dkk_per_eur.iloc[-1]:.4f}")


The data show why the fixed-rate assumption is a useful approximation for Denmark. The nominal exchange rate moves very little relative to the formal band. This means that short-run real adjustment must come mainly from inflation differentials, not from nominal depreciation.

The intuition is that the peg removes one usual adjustment margin. If the krone is kept close to the euro, Danish goods cannot become cheaper through a nominal depreciation against the euro. Competitiveness therefore has to move through relative prices: Danish inflation below euro-area inflation makes Danish goods cheaper over time.


## 3. <a id='toc3_'></a>[UIP and credibility](#toc0_)

Under a credible peg, expected depreciation is close to zero and UIP gives $i_t\approx i_t^f$. If markets expect a devaluation, the expected depreciation term becomes positive and the domestic rate must rise.

This is why credibility matters for macroeconomic stabilisation. With a credible peg, the domestic interest rate is anchored by the foreign rate. With doubts about the peg, investors require compensation for expected currency losses, so the country can face higher interest rates exactly when confidence is weak.


In [ ]:
expected_devaluation = np.array([0.00, 0.005, 0.01, 0.02, 0.05])
uip_table = pd.DataFrame({
    'Expected devaluation (%)': 100 * expected_devaluation,
    'Foreign rate (%)': 100 * par['i_f'],
    'UIP domestic rate (%)': 100 * uip_rate(par['i_f'], expected_devaluation),
})
uip_table


In [ ]:
dep_grid = np.linspace(0, 0.06, 200)
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(100 * dep_grid, 100 * uip_rate(par['i_f'], dep_grid), label='UIP domestic rate')
ax.axhline(100 * par['i_f'], linestyle='--', label='foreign rate')
ax.set_title('Expected devaluation and the domestic interest rate')
ax.set_xlabel('Expected devaluation, percent')
ax.set_ylabel('Nominal interest rate, percent')
ax.legend(loc='upper left')
plt.show()


**Try it yourself.** Change `par['i_f']` to `0.04`, then rerun the table and plot. Does the UIP line change slope or only shift up? Next, compute the domestic rate if the foreign rate is 2% and markets expect a 3% devaluation.


In [ ]:
# your code here


## 4. <a id='toc4_'></a>[Inflation differentials and real adjustment](#toc0_)

Under a fixed nominal exchange rate, $\Delta e_t=0$, so

$$
\Delta e^r_t=\pi^f_t-\pi_t.
$$

If Danish inflation is below euro-area inflation, Danish goods become cheaper relative to euro-area goods. That is a real depreciation even though the nominal exchange rate does not move.

The result is useful because it separates nominal stability from real adjustment. A fixed nominal exchange rate does not freeze competitiveness. It only means that competitiveness changes slowly through cumulative inflation gaps rather than quickly through exchange-rate jumps.


In [ ]:
dk_hicp = read_csv_series('CP0000DKM086NEST.csv')
ea_hicp = read_csv_series('CP00MI15EA20M086NEST.csv')
pi = pd.concat({'Denmark': yoy(dk_hicp), 'Euro area': yoy(ea_hicp)}, axis=1).dropna()
pi = pi.loc[pi.index >= '2021-01-01']

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(pi.index, pi['Denmark'], label='Denmark')
ax.plot(pi.index, pi['Euro area'], linestyle='--', label='Euro area')
ax.axhline(2.0, linestyle=':', label='2% reference')
ax.set_title('HICP inflation: Denmark and the euro area')
ax.set_xlabel('Date')
ax.set_ylabel('Year-on-year inflation, percent')
ax.legend(loc='upper right')
plt.show()


In [ ]:
monthly_diff = (pi['Euro area'] - pi['Denmark']) / 12.0
rer_index = 100.0 + monthly_diff.cumsum()

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(rer_index.index, rer_index, label='Approx. real exchange rate')
ax.axhline(100.0, linestyle='--', label='Jan. 2021 = 100')
ax.set_title('Real-exchange-rate drift from inflation differentials')
ax.set_xlabel('Date')
ax.set_ylabel('Index')
ax.legend(loc='upper left')
plt.show()

print(f"End-of-sample index: {rer_index.iloc[-1]:.2f}")


The index cumulates the monthly approximation $(\pi^f-\pi)/12$. Values above 100 mean that Denmark has gained price competitiveness relative to January 2021. This is the same mechanism as in the AS--AD model: low domestic inflation raises $e^r$, which supports net exports.

The intuition of the plotted index is cumulative. A small monthly inflation gap does not matter much on its own, but repeated gaps compound into a visible change in relative prices. That is why fixed-rate adjustment is often gradual but persistent.


## 5. <a id='toc5_'></a>[Fixed-rate AS--AD dynamics](#toc0_)

A negative demand shock lowers output and inflation. Lower inflation raises the real exchange rate, which gradually shifts AD back to the right.

The useful part of this simulation is that it shows the self-correcting force in a fixed-rate economy. The initial recession is painful, but lower domestic inflation slowly restores competitiveness. The stronger the response of inflation to output, and the stronger demand responds to competitiveness, the faster the model returns toward equilibrium.


In [ ]:
T = par['T']
z_recession = np.zeros(T)
z_recession[0] = -1.0

path = simulate_fixed_rate(T, par['beta1'], par['gamma'], z_path=z_recession)
path.head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(path['period'], path['yhat'], label='output gap')
ax.plot(path['period'], path['pihat'], linestyle='--', label='inflation gap')
ax.plot(path['period'], path['er'], linestyle=':', label='real exchange rate')
ax.axhline(0.0, linewidth=1)
ax.set_title('Adjustment after a one-period negative demand shock')
ax.set_xlabel('Period')
ax.set_ylabel('Deviation from long-run equilibrium')
ax.legend(loc='best')
plt.show()


**Try it yourself.** Lower `par['gamma']` to `0.15` and rerun the recession simulation. Does the economy recover faster or more slowly? Then raise `par['beta1']` to `1.2`. Which parameter matters more for the speed of adjustment in this calibration?


In [ ]:
# your code here


## 6. <a id='toc6_'></a>[Fiscal policy under a peg](#toc0_)

A one-period fiscal expansion raises demand on impact. But the boom raises inflation, and inflation above the foreign rate reduces the real exchange rate. When the fiscal impulse disappears, the economy is left with lower competitiveness.

The intuition is that demand management under a peg has a trade-off over time. Fiscal stimulus can close a gap today, but if it pushes domestic inflation above foreign inflation, it creates a real appreciation. After the stimulus ends, weaker competitiveness can pull output below potential.


In [ ]:
z_stimulus = np.zeros(T)
z_stimulus[0] = 1.0
stimulus_path = simulate_fixed_rate(T, par['beta1'], par['gamma'], z_path=z_stimulus)
stimulus_path.head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(stimulus_path['period'], stimulus_path['yhat'], label='output gap')
ax.plot(stimulus_path['period'], stimulus_path['pihat'], linestyle='--', label='inflation gap')
ax.plot(stimulus_path['period'], stimulus_path['er'], linestyle=':', label='real exchange rate')
ax.axhline(0.0, linewidth=1)
ax.set_title('Temporary fiscal expansion and competitiveness hangover')
ax.set_xlabel('Period')
ax.set_ylabel('Deviation from long-run equilibrium')
ax.legend(loc='best')
plt.show()

first_negative = stimulus_path.loc[stimulus_path['yhat'] < 0, 'period'].min()
print(f"First below-potential period after the stimulus: t = {first_negative}")


Now compare passive fiscal policy with an automatic stabiliser. The rule is

$$
g_t-\bar g=-\phi \widehat y_t.
$$

A larger $\phi$ dampens demand shocks on impact. It also reduces the force that would otherwise push the economy back toward long-run equilibrium.

This is useful because it makes the stabilisation trade-off visible. A stronger automatic stabiliser cushions the first output fall, but by reducing the fall in inflation it also weakens the real-depreciation channel that would help the economy recover under a fixed exchange rate.


In [ ]:
z_neg = np.zeros(T)
z_neg[0] = -1.0
passive = simulate_fixed_rate(T, par['beta1'], par['gamma'], z_path=z_neg, fiscal_phi=0.0)
active = simulate_fixed_rate(T, par['beta1'], par['gamma'], z_path=z_neg, fiscal_phi=0.8, beta3=par['beta3'])

comparison = pd.DataFrame({
    'period': passive['period'],
    'output_gap_passive': passive['yhat'],
    'output_gap_active_phi_0_8': active['yhat'],
    'fiscal_gap_active': active['g_gap'],
})
comparison.head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(passive['period'], passive['yhat'], label='passive fiscal policy')
ax.plot(active['period'], active['yhat'], linestyle='--', label='countercyclical fiscal rule')
ax.axhline(0.0, linewidth=1)
ax.set_title('Output gap with and without automatic fiscal stabilisation')
ax.set_xlabel('Period')
ax.set_ylabel('Output gap')
ax.legend(loc='best')
plt.show()


**Check.** In the active case, the output drop on impact is smaller. But once the economy starts recovering, fiscal policy tightens automatically because the output gap becomes less negative. This is why the rule can slow convergence even while it reduces the initial displacement.

The result should not be read as saying that stabilisers are bad. It says that in this stripped-down model they insure the economy against the first hit, while also muting some of the price adjustment that restores competitiveness.


## 7. <a id='toc7_'></a>[Anticipated devaluation](#toc0_)

If a devaluation is expected, UIP raises the domestic interest rate before the parity change. This is the anticipation effect.

This exercise is useful because it shows why an announced or widely expected devaluation is not equivalent to a surprise devaluation. Before the exchange-rate change arrives, expected depreciation raises the domestic interest rate and can contract demand.


In [ ]:
probs = np.array([0.0, 0.1, 0.25, 0.5, 0.75])
devaluation_size = 0.08   # 8 percent devaluation if it happens
expected_dep = probs * devaluation_size

anticipation = pd.DataFrame({
    'Probability of devaluation': probs,
    'Devaluation size if it happens (%)': 100 * devaluation_size,
    'Expected depreciation (%)': 100 * expected_dep,
    'UIP domestic rate (%)': 100 * uip_rate(par['i_f'], expected_dep),
})
anticipation


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(100 * probs, 100 * uip_rate(par['i_f'], expected_dep), marker='o')
ax.axhline(100 * par['i_f'], linestyle='--', label='foreign rate')
ax.set_title('Probability of devaluation and the domestic rate')
ax.set_xlabel('Probability of devaluation, percent')
ax.set_ylabel('UIP-implied domestic rate, percent')
ax.legend(loc='upper left')
plt.show()


The expected depreciation term is enough to raise the domestic rate. This can depress demand before the devaluation occurs, which is why anticipated devaluation is weaker as a stabilisation tool than a surprise devaluation.


## 8. <a id='toc8_'></a>[Group exercise](#toc0_)

Work in pairs or small groups. Use the code above and write short answers.

1. **UIP.** Let the foreign rate be 2%. Compute the domestic rate under a fully credible peg and under a 2% expected devaluation. Explain the difference in one sentence.
2. **Real exchange rate.** Pick a three-month window in the HICP plot where Denmark gained competitiveness. Approximate the gain using $(\pi^f-\pi)/12$.
3. **Fiscal stimulus.** In the stimulus simulation, why does the economy fall below potential after the first period?
4. **Automatic stabiliser.** Raise `fiscal_phi` to 1.5. Does the initial output gap shrink further? What happens to the later return to zero?
5. **Policy interpretation.** Why is fiscal policy more central under a peg than under an independent monetary policy regime?


In [ ]:
# Workspace for the group exercise


## 9. <a id='toc9_'></a>[Summary](#toc0_)

| Object | Equation | Mechanism |
|---|---|---|
| UIP | $i_t=i_t^f+\Delta e^e_{t+1}$ | credibility pins the domestic rate to the foreign rate |
| Real exchange rate | $\Delta e^r_t=\Delta e_t+\pi^f_t-\pi_t$ | under a peg, inflation differentials do the adjustment |
| AD under peg | $\widehat y_t=\beta_1(e^r_{t-1}-\widehat\pi_t)+z_t$ | competitiveness shifts demand |
| AS | $\widehat\pi_t=\gamma\widehat y_t+s_t$ | output gaps move inflation |
| RER update | $e^r_t=e^r_{t-1}-\widehat\pi_t$ | low inflation creates real depreciation |

The central mechanism is symmetric: recessions generate low inflation and a gradual real depreciation; booms generate high inflation and a gradual real appreciation. Under a fixed nominal exchange rate, this is the core medium-run adjustment channel.

The broader intuition is that a peg shifts the burden of adjustment away from monetary policy and the nominal exchange rate. Output, inflation, competitiveness, and fiscal policy become tightly linked, so a policy that looks stabilising on impact can have delayed real-exchange-rate effects.
